# Bot detection

### Full pr_comments dataset

In [1]:
import pandas as pd

# Loading pr_comments.parquet
pr_comments = pd.read_parquet("Data/pr_comments.parquet")

# Filter user_type=="User"
user_filter = pr_comments['user_type'] == "User"

# Filter potential bots by username
bot_like_filter = pr_comments['user'].str.contains(r'bot|ci|copilot|ai|automation', case=False, na=False)

# Combine filters
suspected_bots = pr_comments[user_filter & bot_like_filter]

# Count the number of comments for each user
bot_counts = suspected_bots['user'].value_counts()

# Filter those who have a minimum of 10 comments
bot_counts_10plus = bot_counts[bot_counts >= 10]

# Sort records by comment count
bot_counts_10plus_sorted = bot_counts_10plus.sort_values(ascending=False)

# Display the first 50 suspects
display(bot_counts_10plus.head(50))

# Find Bots and change to BotOther
bots_users = ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot', 'facebook-github-bot','pulumi-bot','raycastbot']
pr_comments.loc[pr_comments['user'].isin(bots_users), 'user_type'] = 'BotOther'

user
captainsafia       52
jakebailey         48
ian-at-airbyte     34
NoritakaIkeda      32
k8s-ci-robot       28
MayaRainer         27
wolfSSL-Bot        25
vdaas-ci           25
jairmyree          15
hairyhum           13
raineorshine       13
takenagain         12
pytorchmergebot    11
aishwaryabh        11
owidbot            10
Name: count, dtype: int64

### Full pr_comments dataset - weaker bots_filter

In [2]:
filter = pr_comments['user_type']=='User'
bots_filter = pr_comments['user'].str.contains("bot")

display(pr_comments[filter& bots_filter]['user'].value_counts())
display(pr_comments[filter]['user'].value_counts()[:50])

bots_users =  ['CLAassistant', 'codecov-commenter', 'pytorchmergebot', 'k8s-ci-robot', 'robobun', 'wolfSSL-Bot', 'vdaas-ci', 'lightdash-bot', 'typescript-bot', 'calcom-bot', 'azure-sdk', 'owidbot', 'modularbot','facebook-github-bot','pulumi-bot','raycastbot']

pr_comments[pr_comments['user'].isin(bots_users)]

pr_comments.loc[pr_comments['user'].isin(bots_users), 'user_type'] = 'BotOther'

user
metamaskbot                   9
k8s-infra-cherrypick-robot    4
zwave-js-bot                  4
Issues-translate-bot          2
slangbot                      2
gravity-ui-bot                2
botberry                      2
gradio-pr-bot                 2
nf-core-bot                   2
mm-cloud-bot                  1
sre-ci-robot                  1
pyvista-bot                   1
throbotaith                   1
wheelsbot7                    1
istio-policy-bot              1
bits-bot                      1
Name: count, dtype: int64

user
joaomdmoura                            264
davidfowl                              161
anupriya13                             153
PureWeen                               144
slavingia                              132
coveralls                              129
benbalter                              108
vs-mobiletools-engineering-service2     86
mitchdenny                              84
marrobi                                 79
tomhrr                                  70
harupy                                  70
pierrevalade                            69
parseplatformorg                        67
jcstein                                 64
transphorm                              58
shige                                   56
pelikhan                                53
brianrob                                52
T-Gro                                   52
captainsafia                            52
JoshLove-msft                           51
meganrogge                              50
aarons

# Empty PRs

In [19]:
# Loading all PRs from the full dataset
all_pr_df = pd.read_parquet("Data/pull_request.parquet")

# List all
all_pr_ids = all_pr_df["id"].unique()

# PRs with comments
prs_with_comments = pr_comments["pr_id"].unique()

# PRs without comments
prs_no_comments = set(all_pr_ids) - set(prs_with_comments)

len(prs_no_comments)

20621

# Split of PR reviews based on who performs the review

In [5]:
types_per_pr = (
    pr_comments.groupby("pr_id")["user_type"]
      .apply(lambda s: tuple(sorted(set(s))))
      .reset_index(name="type_set")
)

# Count PRs for each set of types
distribution = types_per_pr["type_set"].value_counts()

print(distribution)

print("\n\n# Reviews without human factor")
distribution[distribution.index.map(lambda t: 'User' in t)].sum()/distribution.sum()

type_set
(Bot,)                   7625
(Bot, User)              3749
(User,)                  1066
(BotOther,)               191
(Bot, BotOther, User)     143
(Bot, BotOther)           142
(BotOther, User)           59
Name: count, dtype: int64


# Reviews without human factor


np.float64(0.38666666666666666)

# Code review done by humans for PRs prepared by AI

In [6]:
import re

AGENT_PATTERN = r"(copilot|coderabbit|coderabbitai|bugbot|gemini|claude|devin|" \
                r"sourcery-ai|greptile|giselles-ai|giselle|pytorchbot|pytorchmergebot|sentry|zwave-js-bot|create-docs-giselles-ai|dotnet-policy-service)"

COMMAND_PATTERN = (
    r"(run|retry|recheck|rebase|review|summari[sz]e|start|continue|fix|resolve|"\
    r"merge|test|format|lint|sign|add|remove|update|rerun|check|verify|resume|"\
    r"generate|create|solve|clean|commit|pack|try|thanks|conflicts)"
)

AGENT_CMD_RE = re.compile(
    rf"""(?:^|\s)
    ([@\\\/]?{AGENT_PATTERN})
    [^.\n]*?
    \b{COMMAND_PATTERN}\b
    |
    \b{COMMAND_PATTERN}\b
    [^.\n]*?
    ([@\/\\]?{AGENT_PATTERN})
    """,
    re.IGNORECASE | re.VERBOSE,
)

STARTS_WITH_AGENT = re.compile(
    rf"""
    ^[@/ ]?{AGENT_PATTERN}
    """,
    re.IGNORECASE | re.VERBOSE,
)

OTHER_COMMANDS = re.compile(
    r"^[/#!@\[\\](azp|check-enforcer|easycla|run|rebase|build-deploy-preview|backport|autoformat|test|bump-version|ci|ba-|format-|poe|"\
    r"cherrypick|huppy|poetry-|ok-|queue-|ci-|fix-|engineer|deploypr|update-|format|checkenforcer|approve-|review|gogoeditdiff"\
    r"|microsoft-github-policy-service|intelci|gpu|spinwick)",
    re.IGNORECASE | re.VERBOSE,
)
OTHER_COMMANDS_EXPL = re.compile(
    r"^(<\!-- buildkite-pr-bot-trigger-comment -->|\*\*\[chatops:help\]\*\*|\n\[\!\[coverage status\]|pre-commit\.ci autofix)", re.IGNORECASE)

WORKFLOW = re.compile(r"^\s*(@\w+[\/\w\-]*\s+)?("\
    # --- Status/Closure/Merge Commands ---
    "resolve merge conflict(s)?"\
    "|merge (?:with master|main)"\
    "|closing (?:as stale|out prs created during experiments with devin\.|, integrated the temp fix in https?:\/\/\S+)"\
    "|this pull request was created as a test and will now be closed\.\s*thank you!?"\
    "|proactively closing this pull request\.\s*reopen if you need to after review\.\s*for any questions ping jwilcox\."\
    "|this pull request has been marked as stale due to inactivity\.\s*please update it to keep it open\."\
    "|this pull request has been closed due to lack of response\."\
    "|force-merging, per conversation in slack\.\s*future iterations can resolve the test failures and\/or lack of robust integration test environment\."\
    "|superseded by (?:pr #\d+|https?:\/\/\S+)"\
    # --- Simple Bot/Action Commands ---
    "|lgtm!?"\
    "|ok"\
    "|done"\
    "|good to go\."\
    "|good bot"\
    "|thank you(?: for|!)"\
    "|thanks"\
    "|thx"\
    "|crap"\
    "|nope"\
    "|bruh"\
    "|\!sync"\
    "|recheck"\
    "|try (?:again|this)"\
    "|please (?:try|retry|f)"\
    "|please test your latest changes by seeing if you're able to build\s*src\/core\/src\/core\.csproj"\
    "|can you retry working on this pr\?"\
    "|wake the fuck up samurai, we have prs to merge"\
    # --- Fix/CI Commands ---
    "|fix lint (?:issue|failure(?: in ci)?)"\
    "|lint (?:failure|ing fails)"\
    "|fix merge issues"\
    "|fix failing (?:tests|spec tests)\."\
    "|run (?:build|yarn change\" prerelease, |yarn lint:fix\" |yarn format\"|poetry lock and push it up|docs-build)"\
    "|make sure that the rust formatter has been run against all changed files, as this is currently causing the build to fail\."\
    "|i've fix with \[\w+\]\(https?:\/\/\S+\)"\
    "|revert all changes in the archive directory"\
    "|please remove the files under \.\/vendor\/\*"\
    "|a bunch of files have been added due to some whitespaces, clean out them and only include the files that have changes relevant to this issue\."\
    "|i think something went wrong in your last session, it seems like your changes weren't committed\. let's try again: please verify all of your changes and that the code works by running the json tests in stable and experimental mode"\
    "|it looks like there are still compile errors\.\s*can you try again\?"\
    "|please format the code and commit it back to the pr"\
    # --- Bot/System Status & Generic Error Messages ---
    "|build is still failing"\
    "|test are failing"\
    "|not working"\
    "|doesn't work"\
    "|the snippets failure is related to a preview feature\."\
    "|looks like your session timed out\. try again\."\
    "|already solved"\
    "|not a good solution"\
    "|the agent was blocked by configuration issues from accessing the necessary dependencies to successfully build and test\.\s*those are being fixed and we'll continue experimenting\."\
    "|unbelievable\. you, \[subject name here\], must be the pride of \[subject hometown here\]!"\
    "|it looks like the rebase was not correct\.\s*please start over with a fresh checkout from master, and make sure the build actually finishes \(\w*\) and that the new test actually runs and passes\.\s*even if the code looks simple the pr will not be accepted if the test doesn't actually run\."\
    "|can one of the admins verify this patch?"\
    # --- Action Requests on Documentation/Metadata ---
    "|add a bullet point to releasenotes\.md describing the change to our users"\
    "|update change log"\
    "|the description of the pr is not accurate anymore after all the changes, use examples from the new assert\.areequal tests and improve it to show some nice examples to the user\."\
    "|make sure to follow the github pr title structure like we are doing for integrations"\
    # --- Specific File Reversions ---
    "|revert all changes made to \"publicapi\.unshipped\.txt\" and then reapply changes to those files based on the api changes made in this pr"\
    "|the publicapi\.unshipped\.txt files need to be fixed\s*in order to fix them can you\s*1\) revert all the changes to publicapi\.unshipped\.txt\s*2\) and then make whatever additions you need to make to them in order to fix the build"\
    # --- User/Team Pings and Addressing Issues ---
    "|@\w+[-\/\w]*(?: \([\w\s:]+\))?(?: ping)?"\
    "|address #\d+"\
    "|related to https?:\/\/\S+"\
    "|i have read the cla document and i hereby sign the cla"\
    "|duplicate"\
    "|konfurikuto wo kaishou shite"\
    "|looks good to me"\
    # --- Closing the pattern ---
    ")\s*",
    re.IGNORECASE | re.DOTALL)

AUTO_BOT = re.compile(
    rf"""
    ^\[approvalnotifier\]
    |^\s*hi\s+[@/]?{AGENT_PATTERN}\.\s+thanks\s+for\s+your\s+pr\..*
    |^\s*\*\*?\[WARNING:CONFLICT\]\*\*?.*
    |^\s*\*\*?\[FORMAT\]\*\*?.*
    |^\s*\*\*?API change check\*\*?.*
    |\:?tada\:
    """,
    re.IGNORECASE | re.VERBOSE,
)

def _matches(pattern, text: str) -> bool:
    return isinstance(text, str) and bool(pattern.search(text))


def is_agentic_ai_command(text: str) -> bool:
    return _matches(AGENT_CMD_RE, text) or _matches(STARTS_WITH_AGENT, text)

def is_other_command(text: str) -> bool:
    return _matches(OTHER_COMMANDS, text)

def is_auto_bot(text: str) -> bool:
    return _matches(AUTO_BOT, text) or _matches(OTHER_COMMANDS_EXPL, text)

def is_workflow(text: str) -> bool:
    return _matches(WORKFLOW, text)


def label_rev_type(text: str) -> str:
    if is_agentic_ai_command(text):
        return "agentic_cmd"
    if is_other_command(text) or is_auto_bot(text):
        return "bot"
    if is_workflow(text):
        return "workflow"
    return "review"

<>:45: SyntaxWarning: invalid escape sequence '\.'
<>:46: SyntaxWarning: invalid escape sequence '\.'
<>:47: SyntaxWarning: invalid escape sequence '\.'
<>:48: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\.'
<>:50: SyntaxWarning: invalid escape sequence '\.'
<>:51: SyntaxWarning: invalid escape sequence '\d'
<>:56: SyntaxWarning: invalid escape sequence '\.'
<>:64: SyntaxWarning: invalid escape sequence '\!'
<>:68: SyntaxWarning: invalid escape sequence '\s'
<>:69: SyntaxWarning: invalid escape sequence '\?'
<>:75: SyntaxWarning: invalid escape sequence '\.'
<>:77: SyntaxWarning: invalid escape sequence '\.'
<>:78: SyntaxWarning: invalid escape sequence '\['
<>:80: SyntaxWarning: invalid escape sequence '\.'
<>:81: SyntaxWarning: invalid escape sequence '\.'
<>:82: SyntaxWarning: invalid escape sequence '\.'
<>:83: SyntaxWarning: invalid escape sequence '\.'
<>:90: SyntaxWarning: invalid escape sequence '\.'
<>:91: SyntaxWarning: invalid e

In [7]:
tests = [
    "/gemini review",
    "/azp",
    "/easycla",
    "review for a testing purpose",
    "@copilot try again",
    "try again @copilot",
    "[approvalnotifier]",
    "hi @copilot. thanks for your pr.",
    "#gogoeditdiff",
    "@microsoft-github-policy-service",
    "<!-- buildkite-pr-bot-trigger-comment -->",
    "**[chatops:help]**",
    "\n[![coverage status]",
    "lgtm",
    "pre-commit.ci autofix",
    "!sync",
    "ok    ",
    "can one of the admins verify this patch?    ",
    "lgtm!",
    "/intelci: run",
    "@pytorchmergebot merge",
    "!gpu",
    "**[WARNING:CONFLICT]**",
    "\\azp",
    "comment :tada:",
    "@user lgtm",
    "try this plan again ",
    "please retry the pull request",
    "thank you for reviewers",
    "thank you!",
    
]

for t in tests:
    print(t,"=>", label_rev_type(t))


/gemini review => agentic_cmd
/azp => bot
/easycla => bot
review for a testing purpose => review
@copilot try again => agentic_cmd
try again @copilot => agentic_cmd
[approvalnotifier] => bot
hi @copilot. thanks for your pr. => bot
#gogoeditdiff => bot
@microsoft-github-policy-service => bot
<!-- buildkite-pr-bot-trigger-comment --> => bot
**[chatops:help]** => bot

[![coverage status] => bot
lgtm => workflow
pre-commit.ci autofix => bot
!sync => workflow
ok     => workflow
can one of the admins verify this patch?     => workflow
lgtm! => workflow
/intelci: run => bot
@pytorchmergebot merge => agentic_cmd
!gpu => bot
**[WARNING:CONFLICT]** => bot
\azp => bot
comment :tada: => bot
@user lgtm => workflow
try this plan again  => workflow
please retry the pull request => workflow
thank you for reviewers => workflow
thank you! => workflow


In [8]:
m_comments = pr_comments["user_type"] == "User"
m_bot = pr_comments["user_type"].isin(["Bot", "BotOther"])

pr_comments = pr_comments.copy()
pr_comments.loc[m_comments,"rev_type"] = pr_comments.loc[m_comments, "body"].apply(label_rev_type)
pr_comments.loc[m_bot, "rev_type"] = (pr_comments.loc[m_bot, "body"].apply(label_rev_type))

In [22]:
print(sum(m_bot))
print(len(pr_comments[m_comments]))

28004
11118


In [10]:
pr_comments['rev_type'].value_counts() / sum(m_comments)

rev_type
review         2.515201
agentic_cmd    0.756611
workflow       0.161540
bot            0.085447
Name: count, dtype: float64

In [11]:
sum(m_comments)

11118

In [18]:
pr_comments__human = pr_comments[m_comments]
pr_comments__human__counts = pr_comments__human['rev_type'].value_counts()
display(pr_comments__human__counts)
print(sum(m_comments))
len(pr_comments) - sum(m_comments)

rev_type
review         5767
agentic_cmd    3154
workflow       1408
bot             789
Name: count, dtype: int64

11118


28004

In [ ]:
pr_comments[pr_comments['rev_type'] == 'bot']

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
339,3126788393,3269032263,buger,14009,User,2025-07-28T11:28:19Z,/review how do you feel about this changes? Ar...,bot
424,2875765090,3057825906,lindy65,13020833,User,2025-05-13T09:38:34Z,/spinwick create --env MM_FEATUREFLAGS_Experim...,bot
747,3121064306,3264933329,sre-ci-robot,56469371,User,2025-07-26T02:59:06Z,[APPROVALNOTIFIER] This PR is **NOT APPROVED**...,bot
752,3048720914,3212218608,coveralls,2354108,User,2025-07-08T12:23:06Z,\n[![Coverage Status](https://coveralls.io/bui...,bot
754,3060259699,3212218608,k8s-ci-robot,20407524,BotOther,2025-07-11T03:37:20Z,[APPROVALNOTIFIER] This PR is **APPROVED**\n\n...,bot
...,...,...,...,...,...,...,...,...
38814,2877718073,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:33:16Z,/bump-version type=minor,bot
38822,2877736008,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:37:23Z,/bump-version,bot
38825,2877745636,3060998787,theyueli,61070669,User,2025-05-13T19:40:27Z,"/bump-version type=patch changelog=""Add suppor...",bot
38827,2877751802,3060998787,devin-ai-integration[bot],158243242,Bot,2025-05-13T19:42:48Z,"/bump-version type=patch changelog=""Add suppor...",bot


In [ ]:
display(pr_comments['pr_id'].unique())

array([3107321792, 3234660269, 3037457814, ..., 2761443934, 3047296744,
       2823956710], shape=(12975,))

In [ ]:
pr_comments[(pr_comments['rev_type'] == 'review') & (pr_comments['user_type'] == 'User')]

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
2,2848667986,3037457814,wilsonccccc,6146503,User,2025-05-03T15:12:37Z,"Hi, thanks for sharing and contributing! Pleas...",review
4,3013998830,3132410695,razimantv,3823215,User,2025-06-27T18:16:05Z,Can you rebase it to the current master? Pleas...,review
7,2720368428,2915784381,nickskalkin,3934579,User,2025-03-13T08:30:47Z,Co-authored-by: Claude \r\n😁,review
9,2721609147,2915784381,damassi,236943,User,2025-03-13T15:09:45Z,> Actually rewrite that commit and remove me a...,review
15,3111630135,3256172444,nilo19,36728755,User,2025-07-24T01:13:29Z,If it's hard to review I can split it into mul...,review
...,...,...,...,...,...,...,...,...
39107,2564708592,2762228850,shreysingla11,62649026,User,2024-12-29T12:25:49Z,"### Code Review Summary\n\nOverall, this PR ma...",review
39110,2693663048,2890563538,joaomdmoura,667063,User,2025-03-03T08:52:22Z,Disclaimer: This review was made by a crew of ...,review
39113,2564173971,2761443934,joaomdmoura,667063,User,2024-12-28T04:37:43Z,Disclaimer: This review was made by a crew of ...,review
39118,2874995243,3047296744,retrogtx,76801862,User,2025-05-13T04:07:09Z,failing tests + does not solve the original is...,review


In [ ]:
def pr_summary(labels):
    unique = set(labels)
    return str(unique)

pr_labels_comment = (
    pr_comments[(pr_comments['user_type'] == 'User')].groupby("pr_id")["rev_type"]
      .apply(pr_summary)
      .reset_index(name="pr_type")
)

In [ ]:
counts_comment = pr_labels_comment["pr_type"].value_counts()
print(counts_comment)

pr_type
{'review'}                                      2328
{'agentic_cmd'}                                  900
{'workflow'}                                     456
{'agentic_cmd', 'review'}                        347
{'workflow', 'review'}                           326
{'bot'}                                          198
{'agentic_cmd', 'workflow', 'review'}             80
{'agentic_cmd', 'bot'}                            60
{'agentic_cmd', 'workflow'}                       56
{'bot', 'review'}                                 52
{'review', 'bot'}                                 48
{'workflow', 'agentic_cmd', 'review'}             41
{'workflow', 'agentic_cmd'}                       24
{'agentic_cmd', 'bot', 'review'}                  23
{'workflow', 'bot', 'review'}                     17
{'review', 'workflow', 'bot'}                     14
{'review', 'agentic_cmd', 'bot'}                  13
{'workflow', 'bot'}                               11
{'review', 'agentic_cmd', 'workflow', 

In [ ]:
# No classical human part
print(counts_comment[~counts_comment.index.str.contains("review")].sum()/counts_comment.sum())

# Only clasical
clasical_filter = counts_comment.index.str.contains("{'review'}")
print(counts_comment[clasical_filter].sum()/counts_comment.sum())

# Classical + other
print(counts_comment[counts_comment.index.str.contains("review") & ~clasical_filter].sum()/counts_comment.sum())

0.34104046242774566
0.4640223240980666
0.19493721347418777


In [ ]:
filtered_comments = pr_comments[
    (pr_comments["user_type"] == "User") &
    (pr_comments["rev_type"] == "review") 
]

top_bodies_comments = (
    filtered_comments["body"].str.lower()
    .value_counts()
    .head(50)     
)

#print(top_bodies_comments)

In [ ]:
# Command for searching suspicious data
filtered_comments[filtered_comments['body'].str.lower().str.contains("")]

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
2,2848667986,3037457814,wilsonccccc,6146503,User,2025-05-03T15:12:37Z,"Hi, thanks for sharing and contributing! Pleas...",review
4,3013998830,3132410695,razimantv,3823215,User,2025-06-27T18:16:05Z,Can you rebase it to the current master? Pleas...,review
7,2720368428,2915784381,nickskalkin,3934579,User,2025-03-13T08:30:47Z,Co-authored-by: Claude \r\n😁,review
9,2721609147,2915784381,damassi,236943,User,2025-03-13T15:09:45Z,> Actually rewrite that commit and remove me a...,review
15,3111630135,3256172444,nilo19,36728755,User,2025-07-24T01:13:29Z,If it's hard to review I can split it into mul...,review
...,...,...,...,...,...,...,...,...
39107,2564708592,2762228850,shreysingla11,62649026,User,2024-12-29T12:25:49Z,"### Code Review Summary\n\nOverall, this PR ma...",review
39110,2693663048,2890563538,joaomdmoura,667063,User,2025-03-03T08:52:22Z,Disclaimer: This review was made by a crew of ...,review
39113,2564173971,2761443934,joaomdmoura,667063,User,2024-12-28T04:37:43Z,Disclaimer: This review was made by a crew of ...,review
39118,2874995243,3047296744,retrogtx,76801862,User,2025-05-13T04:07:09Z,failing tests + does not solve the original is...,review


In [ ]:
slash_counts_comments = (
    pr_comments[pr_comments["body"].str.startswith("/", na=False) & (pr_comments["rev_type"] == 'review')]["body"].value_counts()
)
#print(slash_counts_comments)

In [ ]:
mask_comments = pr_comments["body"].str.startswith("/", na=False) & (pr_comments["rev_type"] == "review")

# extract first token after '/'
first_word_comments = pr_comments.loc[mask_comments, "body"].str.extract(r"^/(\S+)", expand=False)

slash_counts_comments = first_word_comments.value_counts()

#print(slash_counts_comments)

In [ ]:
pr_comments[pr_comments['rev_type'] == 'agentic_cmd']

,id,pr_id,user,user_id,user_type,created_at,body,rev_type
0,2927293042,3107321792,coderabbitai[bot],136622811,Bot,2025-06-01T14:15:35Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
1,3090154270,3234660269,Fank,1900106,User,2025-07-18T17:22:34Z,"claude budget reached, development is on hold ...",agentic_cmd
14,3107769358,3256172444,coderabbitai[bot],136622811,Bot,2025-07-23T12:23:45Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
19,2991504795,3163243464,coderabbitai[bot],136622811,Bot,2025-06-20T12:56:07Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
21,3051526209,3214876564,coderabbitai[bot],136622811,Bot,2025-07-09T07:39:39Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
...,...,...,...,...,...,...,...,...
39059,2722820285,2918508503,coderabbitai[bot],136622811,Bot,2025-03-13T22:27:44Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
39062,2891388779,3073587664,devin-ai-integration[bot],158243242,Bot,2025-05-19T15:09:50Z,Devin is archived and cannot be woken up. Plea...,agentic_cmd
39068,2725444393,2920983723,coderabbitai[bot],136622811,Bot,2025-03-14T18:19:45Z,<!-- This is an auto-generated comment: summar...,agentic_cmd
39089,2942423791,3119572928,devin-ai-integration[bot],158243242,Bot,2025-06-05T01:42:22Z,### 🤖 Devin AI Engineer\n\nI'll be helping wit...,agentic_cmd


# Table

### R_pop

In [ ]:
num_pr = len(pr_comments["pr_id"].unique())
total_reviews = len(pr_comments)

reviews_per_pr = pr_comments.groupby("pr_id").size()
mean_reviews = reviews_per_pr.mean()
median_reviews = reviews_per_pr.median()

rev_counts = pr_comments["rev_type"].value_counts()

category_per_pr = (
    pr_comments.groupby(["pr_id", "rev_type"])
               .size()
               .unstack(fill_value=0)
)

mean_per_category = category_per_pr.mean()
median_per_category = category_per_pr.median()

In [ ]:
summary_table_r_pop = pd.DataFrame({
    "Category": [
        "Number of PRs",
        "Total number of reviews",
        "Evaluative",
        "Agentic Cmd",
        "Bot",
        "Workflow",
    ],
    "Total count": [
        num_pr,
        total_reviews,
        rev_counts.get("review", 0),
        rev_counts.get("agentic_cmd", 0),
        rev_counts.get("bot", 0),
        rev_counts.get("workflow", 0),
    ],
    "Mean per PR": [
        None,
        mean_reviews,
        mean_per_category.get("review", 0),
        mean_per_category.get("agentic_cmd", 0),
        mean_per_category.get("bot", 0),
        mean_per_category.get("workflow", 0),
    ],
    "Median per PR": [
        None,
        median_reviews,
        median_per_category.get("review", 0),
        median_per_category.get("agentic_cmd", 0),
        median_per_category.get("bot", 0),
        median_per_category.get("workflow", 0),
    ]
})

summary_table_r_pop

,Category,Total count,Mean per PR,Median per PR
0,Number of PRs,12975,NaN,NaN
1,Total number of reviews,39122,3.015183,2.0
2,Evaluative,27834,2.145202,2.0
3,Agentic Cmd,8412,0.648324,0.0
4,Bot,950,0.073218,0.0
5,Workflow,1926,0.148439,0.0
